# Reverse Leakage in Label-Free Concept Bottleneck Models (LF-CBM)

This notebook investigates whether **concept detectors in Label-Free CBMs are influenced by species identity**,
violating the assumed causal direction:

> concepts → species

We test for **reverse leakage / backwash**:

> species → concepts

We adapt the *recall-gap test* previously used for supervised CBMs to the **label-free setting**,
using the CLIP-derived **concept matrix** as pseudo-ground truth.


## Background: What supervision exists in LF-CBM?

Unlike supervised CBMs, LF-CBM does not have human-labeled concept annotations.

Instead, LF-CBM constructs a **concept matrix**:

$$P_{i,j} = E_I(x_i) \cdot E_T(t_j)$$

where:
- $x_i$ is an image
- $t_j$ is a concept string
- $E_I, E_T$ are CLIP encoders

Each column $P_{:,j}$ represents CLIP's belief about the presence of concept $j$ across the dataset.

**Key observation:** LF-CBM explicitly trains its concept neuron activations to align with $P_{:,j}$.
Therefore, we treat $P$ as **pseudo-ground-truth concept presence**.


In [1]:
# === Assumed inputs ===
# fc: (N, M) LF-CBM concept activations (after W_c)
# P:  (N, M) CLIP concept matrix
# species: (N,) integer species labels
# concept_names: list[str] length M

import numpy as np


## Recall in Supervised CBMs (reference)

For a concept $j$ and species $s$, recall is:

$$\text{Recall}_{s}(j) =
\frac{\sum_{i: s_i=s} \mathbf{1}[\hat y_{i,j}=1 \wedge y^{(gt)}_{i,j}=1]}{\sum_{i: s_i=s} \mathbf{1}[y^{(gt)}_{i,j}=1]}$$

This measures how well the concept detector fires **when the concept is truly present**, restricted to a single species.


## Adapting Recall to LF-CBM

We must define:
- a **pseudo ground truth** concept presence
- a **predicted** concept presence

We use:
- Ground truth: derived from CLIP concept matrix $P$
- Prediction: LF-CBM concept neuron activation $f_c(x)$


### Method 1: Hard-threshold pseudo labels

We define pseudo ground-truth concept presence as:

$$y^{(gt)}_{i,j} = \mathbf{1}[P_{i,j} > \tau_j]$$

where $\tau_j$ is a concept-specific threshold.


In [2]:
def threshold_by_quantile(x, q=0.9):
    """Binary mask for top-q fraction."""
    return x > np.quantile(x, q)


In [3]:
def hard_recall_for_species(
    fc_j, P_j, species, target_species,
    tau_q=0.9, theta_q=0.9
):
    """Hard recall for a single concept j and species s."""
    gt = threshold_by_quantile(P_j, tau_q)
    pred = threshold_by_quantile(fc_j, theta_q)

    mask = (species == target_species)
    tp = np.sum(pred[mask] & gt[mask])
    fn = np.sum((~pred[mask]) & gt[mask])
    return tp / (tp + fn + 1e-8)


### Interpretation (hard recall)

This measures:

> Among images where CLIP thinks concept $j$ is present, how often does the LF-CBM concept neuron fire?

Conditioned on species.

If recall differs significantly across species pairs, this suggests **species-conditioned concept detection**.


### Method 2: Soft / Weighted recall (preferred)

Instead of binarizing CLIP scores, treat them as **continuous evidence**:

$$\text{SoftRecall}_{s}(j) =
\frac{\sum_{i:s_i=s} \hat y_{i,j}\,\tilde P_{i,j}}{\sum_{i:s_i=s} \tilde P_{i,j}}$$

This aligns better with LF-CBM training, which matches **entire activation patterns**, not thresholds.


In [4]:
def soft_recall_for_species(
    fc_j, P_j, species, target_species,
    activation="sigmoid"
):
    """Soft recall for one concept j and species s."""
    mask = (species == target_species)

    # rescale CLIP scores: keep only nonnegative evidence
    P_pos = np.maximum(P_j, 0)

    if activation == "sigmoid":
        pred = 1 / (1 + np.exp(-fc_j))
    else:
        pred = fc_j

    numerator = np.sum(pred[mask] * P_pos[mask])
    denominator = np.sum(P_pos[mask]) + 1e-8
    return numerator / denominator


## Measuring Reverse Leakage: Species Pair Recall Gap

For a concept $j$ and species pair $(s_1, s_2)$, define:

$$\Delta \text{Recall}(j; s_1, s_2) = \text{Recall}_{s_1}(j) - \text{Recall}_{s_2}(j)$$

A significant non-zero gap suggests **species → concept influence**.


In [5]:
def recall_gap(
    fc_j, P_j, species, s1, s2,
    mode="soft"
):
    if mode == "soft":
        r1 = soft_recall_for_species(fc_j, P_j, species, s1)
        r2 = soft_recall_for_species(fc_j, P_j, species, s2)
    else:
        r1 = hard_recall_for_species(fc_j, P_j, species, s1)
        r2 = hard_recall_for_species(fc_j, P_j, species, s2)
    return r1 - r2


## Guardrail: Match CLIP Distributions Across Species (critical)

A recall gap can arise from:
1. True concept prevalence differences across species
2. Species-conditioned detector behavior

To isolate (2), restrict to species pairs where CLIP scores for the concept are matched.
We use a KS-test between the two species' $P_{i,j}$ distributions.


In [6]:
from scipy.stats import ks_2samp

def clip_matched(P_j, species, s1, s2, alpha=0.05):
    """Return True if CLIP distributions appear similar (fail to reject KS-test)."""
    p1 = P_j[species == s1]
    p2 = P_j[species == s2]
    if len(p1) == 0 or len(p2) == 0:
        return False
    _, pval = ks_2samp(p1, p2)
    return pval > alpha


## (Optional) Bootstrap significance test for recall gaps

This tests whether the recall gap is robust to sampling noise.

We resample within each species with replacement and recompute the gap.


In [7]:
def bootstrap_recall_gap(
    fc_j, P_j, species, s1, s2,
    mode="soft",
    n_boot=2000,
    seed=0
):
    rng = np.random.default_rng(seed)

    idx1 = np.where(species == s1)[0]
    idx2 = np.where(species == s2)[0]
    if len(idx1) == 0 or len(idx2) == 0:
        return np.nan, (np.nan, np.nan), np.nan

    gaps = np.empty(n_boot, dtype=float)
    for b in range(n_boot):
        sidx1 = rng.choice(idx1, size=len(idx1), replace=True)
        sidx2 = rng.choice(idx2, size=len(idx2), replace=True)

        # compute recall using resampled sets by masking
        # We'll compute recall by directly using the sampled arrays.
        if mode == "soft":
            r1 = soft_recall_for_species(fc_j[sidx1], P_j[sidx1], np.zeros(len(sidx1), dtype=int), 0)
            r2 = soft_recall_for_species(fc_j[sidx2], P_j[sidx2], np.zeros(len(sidx2), dtype=int), 0)
        else:
            r1 = hard_recall_for_species(fc_j[sidx1], P_j[sidx1], np.zeros(len(sidx1), dtype=int), 0)
            r2 = hard_recall_for_species(fc_j[sidx2], P_j[sidx2], np.zeros(len(sidx2), dtype=int), 0)
        gaps[b] = r1 - r2

    gap_hat = recall_gap(fc_j, P_j, species, s1, s2, mode=mode)
    ci_lo, ci_hi = np.quantile(gaps, [0.025, 0.975])
    # two-sided p-value: how often bootstrap distribution crosses 0 in opposite direction
    p = 2 * min(np.mean(gaps <= 0), np.mean(gaps >= 0))
    return gap_hat, (ci_lo, ci_hi), p


## Summary

- You now have a **principled recall definition** for LF-CBM.
- It directly mirrors the supervised CBM recall-gap test.
- Using $P$ as pseudo-ground-truth is consistent with LF-CBM training.
- Matching CLIP distributions across species helps isolate **species-conditioned concept detection**.
